# XACLE 2027 D01 — Frozen MOE8 score-envelope diagnostics

This notebook reads the paired frozen MOE8 Gate and Full8 feature store. It performs CPU-only Gate inference and D01 diagnostics; it does not train or rebuild components. Train results are descriptive and potentially in-sample. Validation retains the checkpoint selection context. Test predictions are written before public test metadata is opened.

## 1. Kaggle runtime and pinned source

In [ ]:
import hashlib, importlib.util, json, os, subprocess, sys
from pathlib import Path

SOURCE_COMMIT = "8010c4708a023e032ace397e68deb6afa01f56d5"
GATE_SOURCE_COMPAT_COMMIT = "68124d0e57005d29149f659780ab749d14e09532"
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
EXPECTED_COUNTS = {"train": 7500, "validation": 3000, "test": 3000}
EXPECTED_ORDER = [f"expert{i}" for i in range(1, 9)]
EXPECTED_FEATURE_WIDTH = 33032
WORK = Path("/kaggle/working/d01_moe8")
REPORTS, PREDICTIONS, PLOTS = (WORK / name for name in ("reports", "predictions", "plots"))
for folder in (REPORTS, PREDICTIONS, PLOTS):
    folder.mkdir(parents=True, exist_ok=True)
missing = [name for name in ("numpy", "torch", "matplotlib", "yaml") if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Kaggle runtime missing required packages: {missing}")
REPO = Path("/kaggle/working/MOESCORE-D01")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
for commit in (SOURCE_COMMIT, GATE_SOURCE_COMPAT_COMMIT):
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", commit], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == SOURCE_COMMIT
subprocess.run(["git", "-C", str(REPO), "cat-file", "-e", f"{GATE_SOURCE_COMPAT_COMMIT}^{{commit}}"], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print({"source_commit": SOURCE_COMMIT, "device": "cpu"})

## 2. Dataset and checkpoint preflight

In [ ]:
import csv
import numpy as np
import torch
import yaml
from scripts.diagnostics.d01_moe8 import (
    FULL8_GATE_ARCHITECTURE, analyze_split, build_prediction_rows,
    classification_counts_for_score_bin, discover_frozen_artifacts,
    load_gate_checkpoint_for_d01, read_aligned_public_metadata,
    resolve_dataset_mount, summarize_by_generator,
    validate_frozen_artifact_pair, write_prediction_csv,
)
from data.moe_feature_store import preprocessing_contract
from data.moe_training_dataset import MoeTrainingDataset, collate_moe_features
from model.expert_adapters.registry import FULL8_EXPERT_ORDER, profile_contract
from model.moe_gating_core import GatingNetwork
from training.checkpoint_manager import full8_source_code_sha256
from training.moe_evaluator import MoeGateEvaluator

DATASET_IDS = {"artifacts": "qnv8071/moescore-full8-artifacts", "official": "qnv8071/xacle-2027-data"}
ARTIFACTS_ROOT = resolve_dataset_mount(Path("/kaggle/input"), "moescore-full8-artifacts")
OFFICIAL_ROOT = resolve_dataset_mount(Path("/kaggle/input"), "xacle-2027-data")
ARTIFACTS = discover_frozen_artifacts(ARTIFACTS_ROOT)
metadata_candidates = [p for p in OFFICIAL_ROOT.rglob("train_average.csv") if p.parent.name == "meta_data"]
if len(metadata_candidates) != 1:
    raise ValueError(f"Expected one public metadata root under {OFFICIAL_ROOT}; found {metadata_candidates}")
DATA_ROOT = metadata_candidates[0].parent.parent
METADATA_PATHS = {s: DATA_ROOT / "meta_data" / f"{s}_average.csv" for s in EXPECTED_COUNTS}
if any(not p.is_file() for p in METADATA_PATHS.values()):
    raise FileNotFoundError("A public split metadata CSV is missing")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

GATE_RUN_MANIFEST = json.loads(ARTIFACTS["run_manifest"].read_text(encoding="utf-8"))
GATE_CONFIG = yaml.safe_load(ARTIFACTS["gate_config"].read_text(encoding="utf-8"))
if GATE_CONFIG.get("data", {}).get("profile") != "full_8":
    raise ValueError("Attached Gate config is not Full8")
if GATE_RUN_MANIFEST.get("git_commit") != GATE_SOURCE_COMPAT_COMMIT:
    raise ValueError("Gate run commit differs from verified format-7 source")
if not GATE_SOURCE_COMPAT_COMMIT:
    raise ValueError("Gate run manifest has no source commit")
CHECKPOINT_SHA256 = sha256_file(ARTIFACTS["gate_checkpoint"])
if CHECKPOINT_SHA256 != GATE_RUN_MANIFEST.get("gate_checkpoint_sha256"):
    raise ValueError(f"Gate SHA mismatch: file={CHECKPOINT_SHA256}; run={GATE_RUN_MANIFEST.get('gate_checkpoint_sha256')}")
raw_checkpoint = torch.load(ARTIFACTS["gate_checkpoint"], map_location="cpu", weights_only=True)
if not isinstance(raw_checkpoint, dict) or raw_checkpoint.get("format_version") != 7:
    raise ValueError("Expected the paired Full8 format-7 checkpoint")
GATE_CHECKPOINT_COMPAT_SOURCE_HASHES = GATE_RUN_MANIFEST.get("source_code_sha256")
if raw_checkpoint.get("source_code_sha256") != GATE_CHECKPOINT_COMPAT_SOURCE_HASHES:
    raise ValueError("Checkpoint and run-manifest source hashes differ")
if raw_checkpoint.get("expert_checkpoint_provenance") != GATE_RUN_MANIFEST.get("expert_checkpoint_provenance"):
    raise ValueError("Checkpoint and run-manifest expert provenance differ")
verified_source_hashes = {}
for name in GATE_CHECKPOINT_COMPAT_SOURCE_HASHES:
    relative = Path(name)
    if relative.is_absolute() or ".." in relative.parts or not name:
        raise ValueError(f"Unsafe checkpoint source path: {name!r}")
    source = subprocess.check_output([
        "git", "-C", str(REPO), "show", f"{GATE_SOURCE_COMPAT_COMMIT}:{name}"
    ])
    verified_source_hashes[name] = hashlib.sha256(source).hexdigest()
if verified_source_hashes != GATE_CHECKPOINT_COMPAT_SOURCE_HASHES:
    raise ValueError("Checkpoint source hashes differ from the original verified source commit")
ACTIVE_GATE_SOURCE_FILES = (
    "model/moe_gating_core.py", "training/moe_evaluator.py", "training/target_transform.py",
    "data/moe_feature_store.py", "data/full8_contract.py", "model/expert_adapters/registry.py",
)
current_sources = full8_source_code_sha256()
active_differences = {
    name: {"checkpoint": GATE_CHECKPOINT_COMPAT_SOURCE_HASHES[name], "runtime": current_sources.get(name)}
    for name in ACTIVE_GATE_SOURCE_FILES
    if current_sources.get(name) != GATE_CHECKPOINT_COMPAT_SOURCE_HASHES[name]
}
if active_differences:
    raise ValueError(f"Active Gate source differs from checkpoint source: {active_differences}")
del raw_checkpoint
GATE = GatingNetwork(input_feature_dim=33032, num_experts=8, profile="full_8")
GATE_PAYLOAD, GATE_TRAINING_OVERLAP, GATE_SOURCE_AUDIT, GATE_LOAD_ORIGIN = load_gate_checkpoint_for_d01(
    ARTIFACTS["gate_checkpoint"], gate=GATE,
    expected_expert_checkpoint_provenance=GATE_RUN_MANIFEST["expert_checkpoint_provenance"],
    checkpoint_compat_source_hashes=GATE_CHECKPOINT_COMPAT_SOURCE_HASHES,
)
GATE.eval()
architecture = {"input_width": GATE.input_feature_dim, "hidden_width": 512,
                "normalization": "LayerNorm", "activation": "ReLU", "dropout": 0.1,
                "output_width": GATE.num_experts, "output": "Softmax(dim=-1)"}
assert architecture == FULL8_GATE_ARCHITECTURE
print({"artifact_dataset": DATASET_IDS["artifacts"], "official_dataset": DATASET_IDS["official"],
       "checkpoint_sha256": CHECKPOINT_SHA256, "format": GATE_PAYLOAD["format_version"],
       "gate_architecture": architecture, "strict_load": True})

## 3. Validate Full8 splits, shards and official label alignment

In [ ]:
if tuple(FULL8_EXPERT_ORDER) != tuple(EXPECTED_ORDER):
    raise ValueError(f"Unexpected expert ordering: {FULL8_EXPERT_ORDER}")
_, dimensions, width = profile_contract("full_8")
if width != EXPECTED_FEATURE_WIDTH or len(dimensions) != 8:
    raise ValueError("Repository Full8 feature contract is incompatible")
FULL8_ROOT = ARTIFACTS["feature_store"]
FULL8_DATASETS = {s: MoeTrainingDataset(str(FULL8_ROOT), split=s, require_labels=s != "test")
                  for s in EXPECTED_COUNTS}
FULL8_MANIFESTS = {s: dataset.manifest for s, dataset in FULL8_DATASETS.items()}
PREPROCESSING_CONTRACTS = {s: preprocessing_contract(m) for s, m in FULL8_MANIFESTS.items()}
validate_frozen_artifact_pair(
    CHECKPOINT_SHA256, GATE_PAYLOAD, GATE_RUN_MANIFEST, FULL8_MANIFESTS,
    verified_source_hashes=verified_source_hashes, preprocessing_contracts=PREPROCESSING_CONTRACTS,
)
SPLIT_IDENTITIES, OFFICIAL_METADATA = {}, {}
for split, dataset in FULL8_DATASETS.items():
    ids = dataset.store.sample_ids
    names = [name for shard in dataset.manifest["shards"] for name in shard["wave_file_names"]]
    if len(ids) != EXPECTED_COUNTS[split] or ids != [
        sid for shard in dataset.manifest["shards"] for sid in shard["sample_ids"]
    ]:
        raise ValueError(f"{split}: dataset reader changed sample order/count")
    SPLIT_IDENTITIES[split] = {"sample_ids": ids, "wave_file_names": names}
    if split != "test":
        metadata = read_aligned_public_metadata(METADATA_PATHS[split], split, ids, names, include_labels=True)
        if metadata["labels_raw"] is None:
            raise ValueError(f"Official {split} labels are required for D01 diagnostics")
        OFFICIAL_METADATA[split] = metadata
ARTIFACT_VALIDATION = {
    "status": "PASS", "checkpoint_sha256": CHECKPOINT_SHA256,
    "checkpoint_format": GATE_PAYLOAD["format_version"], "gate_architecture": architecture,
    "expert_order": EXPECTED_ORDER, "feature_width": width, "score_width": 8,
    "sample_counts": {s: len(ds) for s, ds in FULL8_DATASETS.items()},
    "store_identities": {s: m["store_identity_sha256"] for s, m in FULL8_MANIFESTS.items()},
    "source_manifest_sha256_by_split": {s: m["source_manifest_sha256"] for s, m in FULL8_MANIFESTS.items()},
    "preprocessing_contracts_match": True, "expert_provenance_matches": True,
    "all_feature_shard_hashes_and_arrays_validated": True, "sample_order_validated": True,
}
(REPORTS / "artifact_validation.json").write_text(json.dumps(ARTIFACT_VALIDATION, indent=2) + "\n")
print(json.dumps(ARTIFACT_VALIDATION, indent=2))

## 4. Frozen CPU Gate inference; freeze test predictions first

In [ ]:
device = torch.device("cpu")
GATE = GATE.to(device)
GATE.eval()
evaluator = MoeGateEvaluator(GATE)
split_outputs = {}
for split, dataset in FULL8_DATASETS.items():
    loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0,
                                         collate_fn=collate_moe_features)
    ids, names, scores_out, weights_out, predictions_out, labels_out = [], [], [], [], [], []
    with torch.inference_mode():
        for batch in loader:
            prediction, weights, scores, _ = evaluator._batch_outputs(batch)
            ids.extend(batch["sample_ids"]); names.extend(batch["wave_file_names"])
            scores_out.append(scores.cpu().numpy()); weights_out.append(weights.cpu().numpy())
            predictions_out.append(prediction.cpu().numpy())
            if "labels_raw" in batch:
                labels_out.append(batch["labels_raw"].cpu().numpy())
    scores = np.concatenate(scores_out).astype(np.float64)
    weights = np.concatenate(weights_out).astype(np.float64)
    prediction = np.concatenate(predictions_out).astype(np.float64)
    if ids != SPLIT_IDENTITIES[split]["sample_ids"] or names != SPLIT_IDENTITIES[split]["wave_file_names"]:
        raise ValueError(f"{split}: frozen output order differs from the Full8 store")
    if scores.shape != (EXPECTED_COUNTS[split], 8) or weights.shape != scores.shape:
        raise ValueError(f"{split}: invalid expert/gate output dimensions")
    if not np.isfinite(scores).all() or not np.isfinite(weights).all() or not np.isfinite(prediction).all():
        raise ValueError(f"{split}: frozen inference produced non-finite values")
    if (weights < -1e-7).any() or not np.allclose(weights.sum(1), 1.0, atol=1e-5, rtol=0):
        raise ValueError(f"{split}: gate weights violate the simplex contract")
    if not np.allclose(prediction, np.einsum("ij,ij->i", scores, weights), atol=2e-6, rtol=2e-6):
        raise ValueError(f"{split}: prediction differs from the expert weighted sum")
    if (prediction < scores.min(1)-2e-6).any() or (prediction > scores.max(1)+2e-6).any():
        raise ValueError(f"{split}: prediction is outside its convex expert interval")
    store_labels = np.concatenate(labels_out).astype(np.float64) if labels_out else None
    if split != "test":
        official = np.asarray(OFFICIAL_METADATA[split]["labels_raw"], dtype=np.float64)
        if store_labels is None or not np.allclose(store_labels, official, rtol=0, atol=1e-5):
            raise ValueError(f"{split}: Full8 labels differ from official metadata")
        labels, metadata = store_labels, OFFICIAL_METADATA[split]
    else:
        if store_labels is not None or dataset.manifest["labels_present"]:
            raise ValueError("Public test Store must be label-free during inference")
        labels, metadata = None, {"model_ids": None, "model_names": None}
    split_outputs[split] = {
        "ids": ids, "wave_file_names": names, "scores": scores, "weights": weights,
        "prediction": prediction,
        "analysis": analyze_split(scores, weights, prediction, labels_raw=labels),
        "metadata": metadata,
    }

test = split_outputs["test"]
test_rows = build_prediction_rows(test["ids"], "test", test["scores"], test["weights"], test["analysis"],
                                  wave_file_names=test["wave_file_names"])
write_prediction_csv(PREDICTIONS / "test_d01.csv", test_rows)
TEST_PREDICTION_SHA256 = sha256_file(PREDICTIONS / "test_d01.csv")
for split in ("train", "validation"):
    data = split_outputs[split]
    rows = build_prediction_rows(
        data["ids"], split, data["scores"], data["weights"], data["analysis"],
        wave_file_names=data["wave_file_names"], labels_raw=data["analysis"]["envelope"]["label_raw"],
        model_ids=data["metadata"]["model_ids"], model_names=data["metadata"]["model_names"],
    )
    write_prediction_csv(PREDICTIONS / f"{split}_d01.csv", rows)
print({"device": "cpu", "frozen_test_prediction_sha256": TEST_PREDICTION_SHA256,
       "sample_counts": {s: len(d["ids"]) for s, d in split_outputs.items()}})

## 5. Optional test labels and D01 reports

In [ ]:
test = split_outputs["test"]
test_metadata = read_aligned_public_metadata(
    METADATA_PATHS["test"], "test", test["ids"], test["wave_file_names"], include_labels=True,
)
test["metadata"], test["labels_official"] = test_metadata, test_metadata["labels_raw"]
if test["labels_official"] is not None:
    test["analysis"] = analyze_split(test["scores"], test["weights"], test["prediction"],
                                     labels_raw=test["labels_official"])
if sha256_file(PREDICTIONS / "test_d01.csv") != TEST_PREDICTION_SHA256:
    raise RuntimeError("Frozen test predictions changed during optional label evaluation")

from collections import Counter, defaultdict
import itertools
import matplotlib.pyplot as plt

global_rows, bin_rows, generator_rows, structural = [], [], [], {}
for split, data in split_outputs.items():
    analysis = data["analysis"]
    if analysis["global_metrics"] is None:
        global_rows.append({"split": split, "status": "LABELS_UNAVAILABLE", "sample_count": len(data["ids"])})
        generator_rows.append({"split": split, "status": "LABELS_UNAVAILABLE", "sample_count": len(data["ids"])})
        continue
    global_rows.append({"split": split, **analysis["global_metrics"]})
    bin_rows.extend({"split": split, **row} for row in analysis["per_bin_metrics"])
    structural[split] = analysis["structural_vs_gating"]
    generator_rows.extend({"split": split, **row} for row in summarize_by_generator(
        analysis, generator_ids=data["metadata"]["model_ids"], generator_names=data["metadata"]["model_names"]))
def write_dict_csv(path, rows):
    if not rows:
        Path(path).write_text("status\nNO_ROWS\n", encoding="utf-8"); return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
        writer.writeheader(); writer.writerows(rows)
write_dict_csv(REPORTS / "global_metrics.csv", global_rows)
write_dict_csv(REPORTS / "per_bin_metrics.csv", bin_rows)
write_dict_csv(REPORTS / "generator_metrics.csv", generator_rows or [{"status": "GENERATOR_METADATA_UNAVAILABLE"}])
(REPORTS / "structural_vs_gating.json").write_text(json.dumps(structural, indent=2, allow_nan=False) + "\n")

# Overlap is a descriptive annotation, never a gate or sample exclusion.
groups, affected = defaultdict(list), {s: set() for s in EXPECTED_COUNTS}
for split, manifest in FULL8_MANIFESTS.items():
    for row in manifest["assembly_identity"]["ordered_source_fingerprints"]:
        groups[row["wav_sha256"].lower()].append({"split": split, "sample_id": row["sample_id"]})
overlap_groups, pair_counts = [], Counter()
for digest, rows in groups.items():
    split_names = sorted({row["split"] for row in rows})
    if len(split_names) < 2: continue
    for first, second in itertools.combinations(split_names, 2): pair_counts[f"{first}/{second}"] += 1
    for row in rows: affected[row["split"]].add(row["sample_id"])
    overlap_groups.append({"wav_sha256": digest, "samples": rows})
overlap = {
    "total_overlapping_sha256_groups": len(overlap_groups), "pair_overlap_group_counts": dict(pair_counts),
    "affected_sample_ids_by_split": {s: sorted(v) for s, v in affected.items()},
    "groups": overlap_groups, "active_policy": "descriptive_only_all_public_rows_retained",
    "evaluation_bias_warning": "Cross-split identical WAVs may bias split-level evaluation.",
    "retained_counts": {s: len(d["ids"]) for s, d in split_outputs.items()},
}
(REPORTS / "audio_overlap.json").write_text(json.dumps(overlap, indent=2) + "\n")
sensitivity = {}
for split, data in split_outputs.items():
    labels = data["analysis"]["envelope"].get("label_raw")
    keep = np.asarray([sid not in affected[split] for sid in data["ids"]], dtype=bool)
    item = {"split": split, "overlapping_sample_count": int((~keep).sum()),
            "non_overlapping_sample_count": int(keep.sum()), "all_samples_retained_in_predictions": True}
    if labels is None:
        item["status"] = "LABELS_UNAVAILABLE"
    elif keep.any():
        item["status"] = "AVAILABLE"; item["all_sample_metrics"] = data["analysis"]["global_metrics"]
        item["non_overlapping_metrics"] = analyze_split(
            data["scores"][keep], data["weights"][keep], data["prediction"][keep],
            labels_raw=np.asarray(labels)[keep])["global_metrics"]
    else: item["status"] = "NO_NON_OVERLAPPING_SAMPLES"
    sensitivity[split] = item
(REPORTS / "overlap_metric_sensitivity.json").write_text(json.dumps(sensitivity, indent=2) + "\n")
summary = {
    "experiment": "XACLE 2027 D01 Expert Score Envelope Analysis", "source_commit": SOURCE_COMMIT,
    "gate_source_compat_commit": GATE_SOURCE_COMPAT_COMMIT, "expert_order": EXPECTED_ORDER,
    "feature_width": 33032, "split_counts": {s: len(d["ids"]) for s, d in split_outputs.items()},
    "retained_public_sample_count": sum(len(d["ids"]) for d in split_outputs.values()),
    "all_public_samples_retained": sum(len(d["ids"]) for d in split_outputs.values()) == 13500,
    "public_split_audio_overlap": overlap, "overlap_metric_sensitivity": sensitivity,
    "metrics_by_split": {s: d["analysis"]["global_metrics"] for s, d in split_outputs.items()},
    "classification_by_split": {s: d["analysis"]["classification"] for s, d in split_outputs.items()},
    "train_is_in_sample": True, "not_logo": True,
    "validation_selection_policy": GATE_RUN_MANIFEST.get("selection_policy"),
    "test_labels_read_after_predictions_frozen": True,
}
(REPORTS / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n")

plot_source = 'import matplotlib.pyplot as plt\ndef save_plot(fig, filename):\n    fig.tight_layout()\n    fig.savefig(PLOTS / filename, dpi=220, bbox_inches="tight")\n    plt.close(fig)\n\ndef plot_labeled_split(split, data):\n    analysis = data["analysis"]\n    envelope = analysis["envelope"]\n    y = envelope.get("label_raw")\n    if y is None:\n        pred = envelope["moe_prediction_raw"]\n        lo, hi = envelope["expert_min_raw"], envelope["expert_max_raw"]\n        fig, ax = plt.subplots(figsize=(8, 5))\n        ax.hist(pred, bins=np.arange(-10, 20.5, .25), color="darkorange", alpha=.8)\n        ax.set(xlim=(0, 10), xlabel="Raw score interpretation (0–10 display)", ylabel="Sample count",\n               title=f"{split}: label-free MOE8 prediction histogram (n={len(pred)}; GT unavailable)")\n        save_plot(fig, f"{split}_01_hist_prediction_labels_unavailable.png")\n        fig, ax = plt.subplots(figsize=(8, 5)); bins = np.arange(-10, 20.5, .25)\n        ax.hist(lo, bins=bins, histtype="step", linewidth=1.8, label="Expert minimum")\n        ax.hist(hi, bins=bins, histtype="step", linewidth=1.8, label="Expert maximum")\n        ax.set(xlim=(0, 10), xlabel="Raw score interpretation (0–10 display)", ylabel="Count",\n               title=f"{split}: expert envelope endpoint distributions (n={len(pred)}; GT unavailable)")\n        ax.legend(); save_plot(fig, f"{split}_02_hist_expert_endpoints_labels_unavailable.png")\n        fig, ax = plt.subplots(figsize=(8, 5)); ax.hist(envelope["envelope_width_raw"], bins=40, color="slateblue")\n        ax.set(xlabel="Expert envelope width (raw units)", ylabel="Sample count",\n               title=f"{split}: label-free envelope width (n={len(pred)})")\n        save_plot(fig, f"{split}_09_envelope_width_distribution.png")\n        return\n    y = np.asarray(y)\n    pred = envelope["moe_prediction_raw"]\n    lo, hi, width = envelope["expert_min_raw"], envelope["expert_max_raw"], envelope["envelope_width_raw"]\n    tag = f"{split}_"\n    fig, ax = plt.subplots(figsize=(8, 5))\n    ax.hist(y, bins=np.arange(0, 10.5, .25), alpha=.55, label=f"Ground truth (n={len(y)})", density=True)\n    ax.hist(pred, bins=np.arange(-10, 20.5, .25), alpha=.55, label="Frozen MOE8 prediction", density=True)\n    pred_outside = int(np.sum((pred < 0) | (pred > 10)))\n    ax.set(xlim=(0, 10), xlabel="Raw score (0–10)", ylabel="Density", title=f"{split}: ground truth vs MOE8 predictions (n={len(y)}; predictions outside display={pred_outside})")\n    ax.legend(); save_plot(fig, tag + "01_hist_ground_truth_vs_prediction.png")\n\n    fig, ax = plt.subplots(figsize=(8, 5))\n    bins = np.arange(-10, 20.5, .25)\n    ax.hist(y, bins=bins, histtype="step", linewidth=2, label="Ground truth")\n    ax.hist(lo, bins=bins, histtype="step", linewidth=1.8, label="Expert minimum")\n    ax.hist(hi, bins=bins, histtype="step", linewidth=1.8, label="Expert maximum")\n    endpoint_outside = int(np.sum((lo < 0) | (lo > 10)) + np.sum((hi < 0) | (hi > 10)))\n    ax.set(xlim=(0, 10), xlabel="Raw score units (0–10 display)", ylabel="Count", title=f"{split}: target and expert endpoints (n={len(y)}; endpoint values outside display={endpoint_outside})")\n    ax.legend(); save_plot(fig, tag + "02_hist_ground_truth_vs_expert_endpoints.png")\n\n    for values, name, label in ((hi, "03_ground_truth_vs_expert_max.png", "Expert maximum"),\n                                (lo, "04_ground_truth_vs_expert_min.png", "Expert minimum")):\n        fig, ax = plt.subplots(figsize=(6.5, 6))\n        ax.scatter(y, values, s=9, alpha=.35, rasterized=True)\n        ax.plot([0, 10], [0, 10], "k--", linewidth=1)\n        outside_count = int(np.sum((values < 0) | (values > 10)))\n        ax.set(xlim=(0, 10), ylim=(0, 10), xlabel="Ground truth (raw 0–10)", ylabel=f"{label} (raw interpretation)", title=f"{split}: ground truth vs {label.lower()} (n={len(y)}; expert values outside display={outside_count})")\n        save_plot(fig, tag + name)\n\n    fig, ax = plt.subplots(figsize=(6.5, 6))\n    ax.scatter(y, pred, s=9, alpha=.35, rasterized=True)\n    ax.plot([0, 10], [0, 10], "k--", linewidth=1)\n    ax.set(xlim=(0, 10), ylim=(0, 10), xlabel="Ground truth (raw 0–10)", ylabel="Frozen MOE8 prediction (raw)", title=f"{split}: prediction vs target (n={len(y)}; prediction values outside display={int(np.sum((pred < 0) | (pred > 10)))})")\n    save_plot(fig, tag + "05_ground_truth_vs_moe_prediction.png")\n\n    bins = analysis["per_bin_metrics"]\n    labels = [f"{r[\'ground_truth_bin\']}\\nn={r[\'sample_count\']}" for r in bins]\n    for field, title, filename in (("above_expert_max_fraction", "Ground truth above every expert", "06_fraction_above_expert_max.png"),\n                                   ("below_expert_min_fraction", "Ground truth below every expert", "07_fraction_below_expert_min.png")):\n        vals = [r[field] for r in bins]\n        fig, ax = plt.subplots(figsize=(10, 4.5)); ax.bar(labels, [0 if v is None else v for v in vals])\n        ax.set(ylim=(0, 1), ylabel="Fraction", title=f"{split}: {title} by target bin (n={len(y)})")\n        ax.tick_params(axis="x", rotation=35); save_plot(fig, tag + filename)\n\n    fig, ax = plt.subplots(figsize=(10, 4.5)); x = np.arange(len(bins)); bw = .38\n    ax.bar(x - bw/2, [0 if r["convex_mse_lower_bound"] is None else r["convex_mse_lower_bound"] for r in bins], bw, label="Convex oracle MSE lower bound")\n    ax.bar(x + bw/2, [0 if r["actual_moe_mse"] is None else r["actual_moe_mse"] for r in bins], bw, label="Frozen MOE8 MSE")\n    ax.set(xticks=x, xticklabels=labels, ylabel="MSE (raw score²)", title=f"{split}: structural lower bound vs gate error (n={len(y)})")\n    ax.tick_params(axis="x", rotation=35); ax.legend(); save_plot(fig, tag + "08_per_bin_lower_bound_vs_actual_mse.png")\n\n    fig, ax = plt.subplots(figsize=(8, 5)); ax.hist(width, bins=40, color="slateblue", alpha=.8)\n    ax.set(xlabel="Expert envelope width (raw units)", ylabel="Sample count", title=f"{split}: envelope width distribution (n={len(width)})")\n    save_plot(fig, tag + "09_envelope_width_distribution.png")\n\n    tails = analysis["tail_metrics"]\n    fig, ax = plt.subplots(figsize=(9, 5)); x = np.arange(len(tails)); bw = .25\n    for offset, field, label in ((-bw, "outside_envelope_fraction", "Outside envelope"),\n                                 (0, "above_expert_max_fraction", "Above max"),\n                                 (bw, "below_expert_min_fraction", "Below min")):\n        ax.bar(x + offset, [0 if r[field] is None else r[field] for r in tails], bw, label=label)\n    ax.set(xticks=x, xticklabels=[f"{r[\'ground_truth_range\']}\\nn={r[\'sample_count\']}" for r in tails], ylim=(0, 1), ylabel="Fraction", title=f"{split}: low/middle/high tail envelope diagnostics (n={len(y)})")\n    ax.legend(); save_plot(fig, tag + "10_tail_analysis.png")\n    out_of_view = int(np.sum((pred < 0) | (pred > 10)) + np.sum((lo < 0) | (lo > 10)) + np.sum((hi < 0) | (hi > 10)))\n    return out_of_view\n\nplot_display_outliers = {}\nfor split, data in split_outputs.items():\n    plot_display_outliers[split] = plot_labeled_split(split, data)'
exec(compile(plot_source, "d01_plot_helpers.py", "exec"), globals())

# Generator-stratified CSV status and figures when metadata and group size permit.
for split, data in split_outputs.items():
    if data["analysis"]["global_metrics"] is None: continue
    generator_ids = data["metadata"]["model_ids"] or data["metadata"]["model_names"]
    if generator_ids is None: continue
    counts = Counter(generator_ids)
    selected = [r for r in generator_rows if r.get("split") == split and r.get("status") == "AVAILABLE"
                and counts.get(r.get("generator_id"), 0) >= 50]
    if selected:
        fig, ax = plt.subplots(figsize=(max(8, len(selected)*.45), 5))
        ax.bar([f"{r['generator_id']} n={r['sample_count']}" for r in selected],
               [r["outside_envelope_fraction"] for r in selected])
        ax.set(ylim=(0, 1), xlabel="Official generator ID", ylabel="Outside-envelope fraction",
               title=f"{split}: generator diagnostics (n≥50)")
        ax.tick_params(axis="x", rotation=50); save_plot(fig, f"{split}_generator_outside_envelope.png")

score_contracts = FULL8_MANIFESTS["train"]["assembly_identity"]["score_contracts"]
score_audit = {}
for index, expert in enumerate(EXPECTED_ORDER):
    score_audit[expert] = {"contract": score_contracts[expert], "by_split": {}}
    for split, data in split_outputs.items():
        values = data["scores"][:, index]; raw = values * 5 + 5
        score_audit[expert]["by_split"][split] = {
            "normalized_min": float(values.min()), "normalized_max": float(values.max()),
            "fraction_outside_normalized_range": float(np.mean((values < -1) | (values > 1))),
            "raw_interpretation_min": float(raw.min()), "raw_interpretation_max": float(raw.max()),
        }

report = ["# D01 — MOE8: phân tích bao điểm convex", "",
          f"Giữ nguyên toàn bộ {sum(len(d['ids']) for d in split_outputs.values())} mẫu public theo thứ tự store. Không huấn luyện, trích xuất feature hoặc chạy expert. WAV trùng chỉ được ghi nhận; có thể gây bias đánh giá.", "",
          "Không hiệu chỉnh, clip hay chuẩn hóa lại điểm expert. Raw = normalized × 5 + 5; raw MSE = normalized MSE × 25. E1/E2 là similarity proxy, không phải MOS đã hiệu chuẩn. Oracle lower bound là chẩn đoán toán học sample-wise, không đảm bảo Gate đạt được.", "",
          f"Checkpoint SHA-256 {CHECKPOINT_SHA256}, format {GATE_PAYLOAD['format_version']}; source tương thích {GATE_SOURCE_COMPAT_COMMIT}. Train descriptive/in-sample; validation selection policy: {GATE_RUN_MANIFEST.get('selection_policy')}. Không phải LOGO.", ""]
for split, data in split_outputs.items():
    a, m = data["analysis"], data["analysis"]["global_metrics"]
    report += [f"## {split} (n={len(data['ids'])})", ""]
    if m is None:
        report += ["Nhãn public test không khả dụng; không tính oracle hoặc ground-truth metrics.", ""]; continue
    high = next(r for r in a["per_bin_metrics"] if r["ground_truth_bin"] == "[9,10]")
    classes = classification_counts_for_score_bin(a, a["envelope"]["label_raw"], 9, 10, include_upper=True)
    report += [
        f"- Outside={m['outside_envelope_count']}/{m['sample_count']} ({m['outside_envelope_fraction']:.4f}); trên max={m['above_expert_max_count']}; dưới min={m['below_expert_min_count']}.",
        f"- MSE raw²: lower bound={m['convex_mse_lower_bound_raw']:.6f}; MOE8={m['actual_moe_mse_raw']:.6f}; difference={m['actual_moe_mse_minus_lower_bound_raw']:.6f}.",
        f"- [9,10] n={high['sample_count']}; trên max={high['above_expert_max_count']}; dưới min={high['below_expert_min_count']}; inside={high['sample_count']-high['outside_envelope_count']}; C={classes['C']}; D={classes['D']}.", "",
        "| Tail | n | Outside | Lower MSE raw² | MOE8 MSE raw² | Mean width |", "|---|---:|---:|---:|---:|---:|",
    ]
    for row in a["tail_metrics"]:
        report.append(f"| {row['ground_truth_range']} | {row['sample_count']} | {row['outside_envelope_fraction']} | {row['convex_mse_lower_bound_raw']} | {row['actual_moe_mse_raw']} | {row['mean_envelope_width_raw']} |")
    report.append("")
high = next(r for r in split_outputs["validation"]["analysis"]["per_bin_metrics"] if r["ground_truth_bin"] == "[9,10]")
if high["sample_count"] == 0:
    direction = "Validation không có đủ mẫu 9–10 để chọn hướng."
elif high["outside_envelope_count"] > high["sample_count"] - high["outside_envelope_count"]:
    direction = "Validation nghiêng về giới hạn cấu trúc; ưu tiên Direction A (residual/non-convex fusion)."
else:
    direction = "Một phần lớn vùng 9–10 nằm trong envelope; Direction B có tiềm năng, nhưng oracle không bảo đảm Gate đạt mức đó."
report += ["## Hướng tiếp theo", "", direction,
           "Lower bound lớn ủng hộ hạn chế cấu trúc; actual–oracle lớn là dư địa gating tiềm năng, không phải cam kết cải thiện.", "",
           f"Test predictions được lưu trước metadata test, SHA-256 {TEST_PREDICTION_SHA256}. Xem audio_overlap.json; không mẫu nào bị loại.", ""]
(REPORTS / "D01-MOE8-report.md").write_text("\n".join(report), encoding="utf-8")
provenance = {
    "source_commit": SOURCE_COMMIT, "source_repo": REPO_URL, "kaggle_dataset_ids": DATASET_IDS,
    "dataset_versions": "not exposed by attached mount metadata",
    "gate_checkpoint_sha256": CHECKPOINT_SHA256, "checkpoint_format": GATE_PAYLOAD["format_version"],
    "gate_run_commit": GATE_RUN_MANIFEST["git_commit"], "gate_source_compat_commit": GATE_SOURCE_COMPAT_COMMIT,
    "gate_architecture": architecture, "expert_order": EXPECTED_ORDER, "feature_width": width, "score_width": 8,
    "full8_store_identities": {s: m["store_identity_sha256"] for s, m in FULL8_MANIFESTS.items()},
    "source_manifest_sha256_by_split": {s: m["source_manifest_sha256"] for s, m in FULL8_MANIFESTS.items()},
    "preprocessing_contract_identity": PREPROCESSING_CONTRACTS["train"],
    "expert_checkpoint_provenance": GATE_PAYLOAD["expert_checkpoint_provenance"],
    "sample_counts": {s: len(d["ids"]) for s, d in split_outputs.items()},
    "public_overlap_policy": "descriptive_only_all_rows_retained",
    "training_or_extraction_occurred": False, "pretrained_experts_loaded": False,
    "device": "cpu", "strict_checkpoint_state_load": True,
    "test_predictions_frozen_before_test_labels": True,
    "test_prediction_csv_sha256": TEST_PREDICTION_SHA256,
    "active_gate_source_files_match_checkpoint": ACTIVE_GATE_SOURCE_FILES,
    "other_source_differences_not_executed": GATE_SOURCE_AUDIT["different_or_unavailable_files"],
    "expert_score_scale_audit": score_audit,
}
(REPORTS / "provenance.json").write_text(json.dumps(provenance, indent=2, allow_nan=False) + "\n")
print({"output": str(WORK), "plot_count": len(list(PLOTS.glob("*.png")))})

## 6. Interpretation limits

The sample-wise oracle projection is a structural diagnostic, not a learned Gate result. Public samples retain their original store order, including cross-split WAV duplicates. Test ground-truth metrics are reported only when released public labels exist.